# 3D DeepLabCut 工具箱
https://github.com/DeepLabCut/DeepLabCut

本笔记将重点介绍新发布的 3D 项目选项（自 **2.0.7+** 版本起可用）的功能。

我们建议您查看这些命令，然后您可以轻松地修改此笔记以适应您自己的项目。

**总而言之，新功能将实现以下操作：**
- 允许您使用棋盘格图像校准相机
- 计算并修复相机畸变
- 将您的 2D DeepLabCut 跟踪数据转换为 3D
- 可以在 3D 空间中进行绘图

### 创建一个新的 3D 项目：

您每创建一个项目**只运行一次**此函数；项目被定义为给定的一组摄像机和校准图像。您可以随时在该项目中分析新的视频。

`create_new_project_3d` 函数会创建一个新的项目目录，该目录专门用于将 2D 姿态转换为 3D 姿态，同时创建所需的子目录以及一个基础的 3D 项目配置文件。每个项目都由项目名称（例如 `Task1`）、实验者姓名（例如 `YourName`）以及创建日期来唯一标识。

因此，此函数要求用户输入项目名称、实验者姓名以及将要使用的摄像机数量。目前，DeepLabCut 支持使用 2 个摄像机进行三角测量，但在未来的版本中将扩展到支持 2 个以上的摄像机。

可选参数指定了工作目录，即项目目录将被创建的位置。如果未指定可选参数 `working_directory`，则项目目录将在当前工作目录中创建。请注意，3D 项目配置文件的完整路径将被引用为 ``config_path3d``。

注意：请确保您已激活您的 DLC anaconda 环境！

In [4]:
import deeplabcut

In [5]:
#Setup your project variables:
YourName = 'teamDLC'
YourExperimentName = 'testing'

In [8]:
config_path = deeplabcut.create_new_project_3d(YourExperimentName,YourName,num_cameras=2)

Created "/home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples/testing-teamDLC-2019-10-29-3d/camera_matrix"
Created "/home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples/testing-teamDLC-2019-10-29-3d/calibration_images"
Created "/home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples/testing-teamDLC-2019-10-29-3d/undistortion"
Created "/home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples/testing-teamDLC-2019-10-29-3d/corners"
Generated "/home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples/testing-teamDLC-2019-10-29-3d/config.yaml"

A new project with name testing-teamDLC-2019-10-29-3d is created at /home/mackenzie/Desktop/DeepLabCut2.0-2.2/examples and a configurable file (config.yaml) is stored there. If you have not calibrated the cameras, then use the function 'calibrate_camera' to start calibrating the camera otherwise use the function ``triangulate`` to triangulate the dataframe


**提示 1:** 如果您希望将此文件夹放置在当前工作目录之外的某个位置，您也可以传入 `working_directory='工作目录的完整路径'`。

**提示 2:** 您也可以将 `config_path3d` 放在 `deeplabcut.create_new_project_3d` 的前面，以创建一个保存 `config.yaml` 文件路径的变量，即 `config_path3d=deeplabcut.create_new_project_3d(...`

In [9]:
#If you're loading an already created project, just set the 3D Project config_path variable:
#import os
#from pathlib import Path
#config_path3d = os.path.join(os.getcwd(),'testing3D-DeepLabCutTeam-2019-06-05-3d/config.yaml')
#print(config_path3d)

## 校准您的相机！

（**重要**）您必须拍摄棋盘图像来进行相机校准。这里有一些您可以打印并使用的示例棋盘（请将它们固定在平坦、坚硬的表面上！）：https://markhedleyjones.com/projects/calibration-checkerboard-collection。
- 您必须将图像对保存为 **.jpg** 文件。
- 它们的命名应以 `camera-#` 作为前缀，例如，第一对图像应命名为 **camera-1-01.jpg** 和 **camera-2-01.jpg**。
- 在拍摄图像时：
     - 保持棋盘的朝向一致，旋转角度不要超过 30 度。棋盘的圆周旋转会改变不同帧之间的原点（-x, -y 轴的相对关系），可能导致检测到的角点顺序不正确。

     - 覆盖不同的距离，并且在每个距离内，覆盖图像视野的所有部分（所有四个角落和中心）。

     - 尽量使用尽可能大的棋盘，理想情况下应至少包含 8x6 个方格。

     - 目标是拍摄至少 70 对图像，因为在角点检测后，有些图像可能会因为角点检测不正确或检测到的角点顺序错误而被丢弃。
     
#### 示例图像 (DEMO images)：
 
在这里，我们使用了一组标准图像集以及此 Notebook。这些图像是 Matlab 上的 Camera Calibration ToolBox 的一部分；具体来说是示例 5。图像可以从以下链接下载：https://data.caltech.edu/records/20164。下载后，校准图像位于 ../calib_doc/htmls/calib_example.zip 中。


如果您希望运行此 DEMO Notebook，请下载文件并将其放置在 **calibration_images** 目录下。（请注意，第 1 对和第 6 对图像未被正确检测，因此请删除这些图像！）。 


当从您自己的设置中采集校准图像时，建议拍摄多对图像（大约 50-70 对！）。

相机校准是一个**迭代过程**，用户需要选择一组能够正确检测到网格图案的校准图像。函数：``deeplabcut.calibrate_cameras(config_path)`` 会从校准图像中提取网格图案，并将其存储在 `corners` 目录下。网格图案可以是 8x8 或 5x5 等。我们使用 8x6 的网格图案来查找棋盘的内部角点。

在某些情况下，可能会发生角点检测不正确，或者在 camera-1 图像和 camera-2 图像中检测到的角点顺序不正确。我们需要移除这些图像对，因为它们会降低校准的准确性。

**下一步/如果尚未完成：** 请将您的图像（或 DEMO 图像）放入 **calibration_images** 目录中。

### 编辑 `config.yaml` 文件：
- 更改摄像机名称；例如，如果您使用 `"cam1, cam2"`、`"camera-1, camera-2"` 或 `"left, right"` 等命名方式。
- 请注意，一旦设置了这些名称，就**不能再编辑它们**（因为它们会用于后续的其他步骤中）。

In [ ]:
deeplabcut.calibrate_cameras(config_path3d, cbrow =9,cbcol =6,calibrate=False,alpha=0.9)

**注意**：您需要指定棋盘格（checkerboard）有多少行（``cbrow``）和多少列（``cbcol``）（即交点数量——如果不清楚，请参考演示图像）。

另外，首先将变量 ``calibrate`` 设置为 **False**，以便您可以移除任何有问题的图像。您需要目视检查输出以验证检测到的角点，并选择那些角点被正确检测到的图像对。

一旦所有图像集都被选中（即删除文件夹中所有不正确的图像对！），并且角点及其顺序都被正确检测到，那么就可以使用以下方法校准两个相机：

**它们看起来会像下面这样（一个示例帧）：**
<p align="center">
<img src="https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1559776966423-RATM6ZQT8JXHYAN768F6/ke17ZwdGBToddI8pDm48kKmw982fUOZVIQXHUCR1F55Zw-zPPgdn4jUwVcJE1ZvWQUxwkmyExglNqGp0IvTJZUJFbgE-7XRK3dMEBRBhUpw5XnxLBmEFHJGf_0qFdDpmIncOw4kq9OpCHNTYqzGO-E1YJr-Thht9Tdog4YtCwrE/right02_corner.jpg?format=500w

In [ ]:
deeplabcut.calibrate_cameras(config_path3d, cbrow = 9,cbcol = 6, calibrate=True, alpha=0.9)

## 检查去畸变效果：

为了检查立体标定（stereo calibration）的效果如何，建议使用相机矩阵对标定图像和角点进行去畸变处理，然后将这些去畸变后的点投影到去畸变后的图像上，以验证它们是否正确对齐。在 DeepLabCut 中可以按如下方式操作：

In [ ]:
import matplotlib
%matplotlib inline

deeplabcut.check_undistortion(config_path3d)

每个校准图像都会被进行去畸变处理，并保存在 `undistortion` 目录下。同时还会保存一个图表，其中包含一对去畸变后的相机图像，以及叠加在上面的去畸变后的角点。请目视检查此图像。所有校准图像中所有去畸变后的角点都会被三角化，并绘制出来，供用户可视化任何与去畸变相关的错误。如果这些点不正确，请检查并修改校准图像（然后重复校准和此步骤操作）！

## 三角测量法 --> 将你的 2D 转换为 3D！

如果畸变校正中没有错误，那么就可以对来自两个相机的位姿进行三角测量，从而得到 3D DeepLabCut 坐标！

（**关键！**）请以这样的方式命名视频文件：文件名中必须包含在 ``config file``（配置文件）中指定的相机名称。例如，如果相机命名为 ``camera-1`` 和 ``camera-2``（或 ``cam-1``、``cam-2`` 等），那么视频文件名必须包含此命名。例如，它们可以命名为 ``rig-1-mouse-day1-camera-1.avi`` 和 ``rig-1-mouse-day1-camera-2.avi``。值得注意的是，视频的像素尺寸不必相同，但请确保它们与校准图像的尺寸相似（并且必须使用校准时使用的相同相机）。

## （**关键！**）编辑 config.yaml 文件：
你还必须编辑 **3D 项目的 config.yaml** 文件，以指定哪些 DeepLabCut 项目包含 2D 视图的信息。

- 至关重要的是，你需要输入与 2D 项目的 config.yaml 文件中**相同**的身体部位名称。
- 你需要在 2D 配置文件中设置要使用的快照（snapshot）（默认为 -1，即网络训练的最后一个快照）。
- 你需要设置一个“3D 评分器”（scorer 3D）名称；这将指向项目文件，并设置在未来的 3D 输出文件名中。
- 在这里，你也应该定义一个“骨架”（skeleton）（注意，这不是刚性的，它只是在绘图步骤中连接这些点）。并非所有点都需要被“骨架化”（skeletonized），即这些点可以是完整身体部位列表的一个子集。其他点只会绘制到 3D 空间中。

**接下来，** 传入 ``config_path3d`` 和当前的 ``video_path``，后者是存储来自两个摄像机所有视频的**文件夹**路径。您可以通过输入以下命令在 deeplabcut 中执行三角测量：

In [ ]:
# Of course, this does not work on the demo calibration images, 
# but when you are ready for your own dataset, edit and then run the following!

video_path = '/home/yourname/videoFolder'

deeplabcut.triangulate(config_path3d,video_path, videotype='mp4')

现在，**三角化文件**已保存在与视频文件相同的目录下（或者您传入了目标文件夹路径）！这些文件现在可以用于将来的分析。无论何时收集到新视频，都可以运行此步骤，并轻松将其添加到您的自动化分析流程中，例如使用 ``deeplabcut.triangulate(config_path3d, video_path)`` 而不是 ``deeplabcut.analyze_videos``。

## 可视化您的 3D DeepLabCut 视频：

为了在 3D 中可视化姿态，用户可以为特定帧创建 3D 视频（这些文件较大，因此我们建议只查看部分帧）。用户可以指定三角测量文件的路径，并指定起始和结束帧索引来创建 3D 标记视频。请注意，`triangulated_file` 是新创建的文件，该文件的后缀名为 `yourDLC_3D_scorername.h5`。可以使用以下命令来完成此操作：

In [ ]:
deeplabcut.create_labeled_video_3d(config_path,['triangulated_file_folder'],start=50,end=250, trailpoints=3)